# NeuralUX — UX Brain Analyzer

Prediz ativação de regiões cerebrais a partir de screenshots de interface usando representações intermediárias de CLIP ViT-L/14.

**Abordagem:** Diferentes camadas de redes neurais profundas (DNNs) correlacionam com diferentes regiões do córtex visual e cognitivo (Yamins et al., 2014; Schrimpf et al., 2018). Extraímos hidden states e attention maps de cada camada do ViT e mapeamos para 8 ROIs cerebrais.

**Antes de rodar:**
1. `Runtime → Change runtime type → GPU (T4 é suficiente)`
2. Configure os tokens abaixo
3. Rode as células em ordem

In [ ]:
# =============================================
# CONFIGURAÇÃO
# =============================================
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
GIST_ID = userdata.get("GIST_ID") or "397ef638ef931f3ace318e891a741312"
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""

print("GITHUB_TOKEN OK?", bool(GITHUB_TOKEN))
print("GIST_ID:", GIST_ID)
print("ANTHROPIC_API_KEY OK?", bool(os.environ["ANTHROPIC_API_KEY"]))

In [ ]:
%%capture
!apt-get update -qq >/dev/null
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por >/dev/null
!pip install -q transformers torch torchvision gradio opencv-python-headless Pillow matplotlib numpy requests pytesseract

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram:.1f} GB")
else:
    print("Nenhuma GPU — rodando em CPU (mais lento)")

print(f"Device: {device}")

In [ ]:
from transformers import CLIPModel, CLIPProcessor

print("Carregando CLIP ViT-L/14...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14", attn_implementation='eager').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
print("Modelo pronto!")

In [ ]:
import numpy as np
import cv2
from PIL import Image
import os # Adicionado para o teste automatico

# =============================================
# PROBES SEMANTICOS — embeddings de texto CLIP
# para medir similaridade com conceitos especificos
# =============================================
PROBES = {
    "text":     ["text", "words", "labels", "typography", "reading", "headline"],
    "faces":    ["human face", "portrait", "avatar", "person", "profile photo"],
    "layout":   ["grid layout", "organized structure", "navigation menu", "sidebar"],
    "motion":   ["animation", "movement", "dynamic", "transition", "scrolling"],
    "decision": ["button", "call to action", "form", "checkout", "purchase", "subscribe"],
    "semantic": ["meaningful content", "information", "data visualization", "icons with meaning"],
}

probe_embeddings = {}
for key, texts in PROBES.items():
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model.text_model(**inputs)
        emb = clip_model.text_projection(outputs.pooler_output)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        probe_embeddings[key] = emb.mean(dim=0)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

print("Probes semanticos prontos!")


# =============================================
# FUNCOES AUXILIARES
# =============================================
def layer_magnitude(hidden_states, start, end):
    """Magnitude media de ativacao nas camadas [start:end]"""
    # Certifica que temos hidden_states suficientes
    if not hidden_states or start >= len(hidden_states) or end > len(hidden_states):
        return 0.0 # Retorna 0.0 se nao houver camadas ou indices invalidos
    
    layers_to_stack = [hidden_states[i][:, 1:, :] for i in range(start, end) if hidden_states[i] is not None]
    if not layers_to_stack:
        return 0.0
    layers = torch.stack(layers_to_stack)
    return layers.norm(dim=-1).mean().item()

def get_image_embedding(pil_image):
    """Extrai embedding de imagem normalizado via CLIP"""
    inputs = clip_processor(images=pil_image, return_tensors="pt").to(device)
    outputs = clip_model.vision_model(**inputs)
    emb = clip_model.visual_projection(outputs.pooler_output)
    return emb / emb.norm(dim=-1, keepdim=True)

def clip_similarity(image_features, probe_key):
    """Similaridade cosseno entre imagem e probe de texto"""
    probe = probe_embeddings[probe_key].unsqueeze(0)
    probe = probe / probe.norm(dim=-1, keepdim=True)
    return (image_features @ probe.T).item()


# =============================================
# FUNCAO DE EXTRACAO DE FEATURES ROBUSTA
# =============================================
def extract_clip_features(pil_image):
    inputs = clip_processor(images=pil_image, return_tensors="pt").to(device)
    vision_out = clip_model.vision_model(
        **inputs,
        output_hidden_states=True,
        output_attentions=True,
    )
    
    # Handle cases where hidden_states or attentions might be None themselves
    raw_hidden_states = vision_out.hidden_states if vision_out.hidden_states is not None else ()
    raw_attentions = vision_out.attentions if vision_out.attentions is not None else ()

    hs = [h for h in raw_hidden_states if h is not None]
    attn = [a for a in raw_attentions if a is not None]

    # Embedding de imagem via projection layer
    img_features = clip_model.visual_projection(vision_out.pooler_output)
    img_features = img_features / img_features.norm(dim=-1, keepdim=True)
    return hs, attn, img_features


# =============================================
# SCORING POR REGIAO CEREBRAL
# Cada funcao combina features de DNN + analise
# de imagem para estimar ativacao na ROI.
# =============================================

def score_v1(hidden_states, img_cv, n_layers):
    """V1 — Cortex visual primario: bordas, contraste, frequencia espacial.
    Camadas iniciais do ViT correlacionam com V1 (Schrimpf et al., 2018)."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edge_density = cv2.Canny(gray, 50, 150).mean() / 255.0
    contrast = gray.astype(float).std() / 80.0
    laplacian_var = min(cv2.Laplacian(gray, cv2.CV_64F).var() / 2000.0, 1.0)
    # Usar indices relativos para early layers
    early_norm = min(layer_magnitude(hidden_states, 1, n_layers // 4) / 15.0, 1.0)
    raw = 0.30 * edge_density + 0.30 * contrast + 0.25 * laplacian_var + 0.15 * early_norm
    return float(np.clip(raw * 115, 8, 98))


def score_ffa(hidden_states, img_cv, image_features, n_layers):
    """FFA — Area fusiforme de faces: deteccao facial + features mid-ventral.
    Camadas intermediarias (8-14) correspondem ao stream ventral (Yamins et al., 2014)."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    total_pixels = gray.shape[0] * gray.shape[1]
    face_area = sum(w * h for (_, _, w, h) in faces) / total_pixels if len(faces) > 0 else 0
    face_score = min(len(faces) * 0.2 + face_area * 3, 1.0)
    clip_face = (clip_similarity(image_features, "faces") + 1) / 2
    # Usar indices relativos para mid layers
    mid_norm = min(layer_magnitude(hidden_states, n_layers // 3, n_layers * 2 // 3) / 15.0, 1.0)
    raw = 0.50 * face_score + 0.25 * clip_face + 0.25 * mid_norm
    return float(np.clip(raw * 110, 5, 98))


def score_ppa(hidden_states, img_cv, image_features):
    """PPA — Area parahipocampal: layout e estrutura espacial.
    Detecta linhas horizontais/verticais e regularidade espacial."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 80, minLineLength=50, maxLineGap=10)
    hv_lines = 0
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
            if angle < 15 or angle > 165 or 75 < angle < 105:
                hv_lines += 1
    line_score = min(hv_lines / 30.0, 1.0)
    clip_layout = (clip_similarity(image_features, "layout") + 1) / 2
    h, w = gray.shape
    bs = max(h, w) // 8
    if bs > 0:
        blocks = [gray[i:i+bs, j:j+bs].mean() for i in range(0, h - bs, bs) for j in range(0, w - bs, bs)]
        regularity = 1.0 - min(np.std(blocks) / 60.0, 1.0) if blocks else 0.5
    else:
        regularity = 0.5
    raw = 0.40 * line_score + 0.25 * clip_layout + 0.35 * regularity
    return float(np.clip(raw * 115, 8, 98))


def score_v5(hidden_states, img_cv, image_features):
    """V5/MT — Area de movimento: gradientes direcionais e dinamismo implicito.
    Para imagens estaticas, mede elementos que sugerem movimento."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY).astype(float)
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
    grad_energy = min(np.sqrt(sx**2 + sy**2).mean() / 100.0, 1.0)
    angles = np.arctan2(sy, sx) * 180 / np.pi
    diag = (((angles > 30) & (angles < 60)) | ((angles > 120) & (angles < 150))).mean()
    clip_motion = (clip_similarity(image_features, "motion") + 1) / 2
    hue_std = 0
    if len(img_cv.shape) == 3:
        hue_std = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)[:, :, 0].std() / 90.0
    raw = 0.35 * grad_energy + 0.25 * diag * 3 + 0.20 * clip_motion + 0.20 * min(hue_std, 1.0)
    return float(np.clip(raw * 110, 5, 95))


def score_ips(attentions, img_cv, n_attentions):
    """IPS — Sulco intraparietal: atencao visual e foco.
    Usa attention maps do ViT — baixa entropia = atencao focada = IPS alto."""

    # Fallback se nao houver attentions
    if not attentions or n_attentions == 0:
        return float(np.clip(50.0, 8, 98)) # Pontuacao padrao se nao houver attentions

    # Usar indices relativos para late attentions
    # As ultimas 1/4 partes das camadas de atencao
    start_attn_idx = max(0, n_attentions - n_attentions // 4)
    late_attn = torch.stack([attentions[i] for i in range(start_attn_idx, n_attentions) if attentions[i] is not None])

    if late_attn.nelement() == 0: # Verifica se o tensor esta vazio apos empilhar
        return float(np.clip(50.0, 15, 95))

    avg_attn = late_attn.mean(dim=(0, 2))[:, 0, 1:]
    if avg_attn.nelement() == 0:
        return float(np.clip(50.0, 15, 95))

    probs = avg_attn / avg_attn.sum(dim=-1, keepdim=True)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean().item()
    max_entropy = np.log(probs.shape[-1])
    focus = 1.0 - (entropy / max_entropy) if max_entropy > 0 else 0.0 # Evita divisao por zero

    topk_val = probs.topk(k=min(10, probs.shape[-1]), dim=-1).values
    topk = topk_val.sum(dim=-1).mean().item() if topk_val.nelement() > 0 else 0.0

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    sal = np.abs(gray.astype(float) - gray.mean())
    sal_norm = min(sal.std() / max(sal.mean() + 1e-10, 1.0) / 2.0, 1.0)
    raw = 0.35 * focus * 1.5 + 0.35 * topk + 0.3 * sal_norm
    return float(np.clip(raw * 100, 8, 98))


def score_broca(hidden_states, img_cv, image_features, n_layers):
    """Broca — Area de linguagem: processamento de texto e labels.
    Combina CLIP text-probe com deteccao de regioes textuais (MSER)."""
    clip_text = (clip_similarity(image_features, "text") + 1) / 2
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    mser = cv2.MSER_create()
    regions, _ = mser.detectRegions(gray)
    text_regions = 0
    for region in regions:
        x, y, w, h = cv2.boundingRect(region.reshape(-1, 1, 2))
        aspect = w / max(h, 1)
        area = w * h
        if 0.1 < aspect < 10 and 50 < area < 5000:
            text_regions += 1
    text_density = min(text_regions / 200.0, 1.0)
    # Usar indices relativos para late layers
    late_norm = min(layer_magnitude(hidden_states, n_layers * 2 // 3, n_layers - 1) / 15.0, 1.0)
    raw = 0.30 * clip_text + 0.45 * text_density + 0.25 * late_norm
    return float(np.clip(raw * 115, 5, 98))


def score_pfc(hidden_states, image_features, n_layers):
    """PFC — Cortex pre-frontal: tomada de decisao e memoria de trabalho.
    Diversidade de features nas camadas finais indica carga cognitiva."""
    clip_decision = (clip_similarity(image_features, "decision") + 1) / 2

    if not hidden_states or n_layers == 0:
        return float(np.clip(50.0, 8, 98))

    late_features = hidden_states[n_layers - 1][:, 1:, :] if n_layers > 0 and hidden_states[n_layers - 1] is not None else torch.tensor([])
    diversity = min(late_features.std(dim=1).mean().item() / 5.0, 1.0) if late_features.nelement() > 0 else 0.0

    # Usar indices relativos para as ultimas camadas
    start_cls_idx = max(0, n_layers - n_layers // 4)
    last_layers_cls = [hidden_states[i][:, 0, :] for i in range(start_cls_idx, n_layers) if hidden_states[i] is not None and hidden_states[i].shape[1] > 0]

    complexity = 0.0
    if last_layers_cls:
        stacked_cls = torch.stack(last_layers_cls)
        complexity = min(stacked_cls.std(dim=0).mean().item() / 3.0, 1.0) if stacked_cls.nelement() > 0 else 0.0

    raw = 0.25 * clip_decision + 0.40 * diversity + 0.35 * complexity
    return float(np.clip(raw * 110, 8, 98))


def score_semantic(hidden_states, image_features, n_layers):
    """Semantica — Compreensao de contexto e significado.
    Magnitude e ganho semantico entre camadas iniciais e finais."""
    clip_sem = (clip_similarity(image_features, "semantic") + 1) / 2

    if not hidden_states or n_layers == 0:
        return float(np.clip(50.0, 8, 98))

    final_cls = hidden_states[n_layers - 1][:, 0, :] if n_layers > 0 and hidden_states[n_layers - 1] is not None and hidden_states[n_layers - 1].shape[1] > 0 else torch.tensor([])
    sem_mag = min(final_cls.norm(dim=-1).mean().item() / 25.0, 1.0) if final_cls.nelement() > 0 else 0.0

    # Usar indices relativos para early e late CLS tokens
    early_cls = hidden_states[n_layers // 4][:, 0, :] if n_layers // 4 < n_layers and hidden_states[n_layers // 4] is not None and hidden_states[n_layers // 4].shape[1] > 0 else torch.tensor([])
    late_cls = hidden_states[n_layers - 2][:, 0, :] if n_layers - 2 >= 0 and hidden_states[n_layers - 2] is not None and hidden_states[n_layers - 2].shape[1] > 0 else torch.tensor([])

    gain = 0.0
    if early_cls.nelement() > 0 and late_cls.nelement() > 0:
        gain = min((late_cls - early_cls).norm(dim=-1).mean().item() / 20.0, 1.0)

    raw = 0.25 * clip_sem + 0.40 * sem_mag + 0.35 * gain
    return float(np.clip(raw * 110, 8, 98))


# =============================================
# PREDICAO COMPLETA
# =============================================
@torch.no_grad()
def predict_brain_activation(image_path):
    """Prediz ativacao cerebral a partir de um screenshot de interface."""
    pil_image = Image.open(image_path).convert("RGB")
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        # Fallback para caso cv2.imread falhe (e.g., caminho com caracteres especiais ou nao-ascii)
        img_cv = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

    hs, attn, img_features = extract_clip_features(pil_image)

    n_layers = len(hs) # Numero real de hidden states apos filtragem
    n_attentions = len(attn) # Numero real de attention maps apos filtragem

    scores = {
        "Visual primário (V1)":       score_v1(hs, img_cv, n_layers),
        "Faces / avatares (FFA)":     score_ffa(hs, img_cv, img_features, n_layers),
        "Layouts / cenas (PPA)":      score_ppa(hs, img_cv, img_features),
        "Movimento / animação (V5)":  score_v5(hs, img_cv, img_features),
        "Atenção visual (IPS)":       score_ips(attn, img_cv, n_attentions),
        "Texto / labels (Broca)":     score_broca(hs, img_cv, img_features, n_layers),
        "Decisão / memória (PFC)":    score_pfc(hs, img_features, n_layers),
        "Semântica / contexto":       score_semantic(hs, img_features, n_layers),
    }
    return scores


print("Pipeline de analise pronto!")

# =============================================
# TESTE AUTOMATICO (NOVA ADICAO)
# =============================================
print("\nExecutando teste automatico com imagem dummy...")
try:
    # Cria uma imagem dummy (e.g., uma imagem preta 224x224)
    dummy_image_path = "/tmp/dummy_image.png"
    Image.new('RGB', (224, 224), color = 'black').save(dummy_image_path)

    test_scores = predict_brain_activation(dummy_image_path)
    print("Teste automatico concluido com sucesso. Scores:")
    for k, v in test_scores.items():
        print(f"  {k}: {v:.2f}")
except Exception as e:
    print(f"Erro durante o teste automatico: {e}")
finally:
    # Limpa a imagem dummy
    if os.path.exists(dummy_image_path):
        os.remove(dummy_image_path)


In [ ]:
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tempfile, time, json, requests, os, sys, hashlib
import re
from pathlib import Path
import numpy as np

# =============================================
# INTEGRACAO NEUROSCORE V2 (repo local)
# =============================================
def _configure_repo_imports():
    candidates = [
        Path.cwd(),
        Path.cwd() / "neuralux-redirect",
        Path("/content/neuralux-redirect"),
    ]
    for base in candidates:
        if (base / "backend" / "neuroscore_v2.py").exists():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            return str(base)
    return None

REPO_BASE = _configure_repo_imports()
HAS_NEUROSCORE_V2 = False
analyze_media = None

try:
    from backend.neuroscore_v2 import analyze_media as _analyze_media
    analyze_media = _analyze_media
    HAS_NEUROSCORE_V2 = True
    print(f"NeuroScore v2 carregado de: {REPO_BASE or Path.cwd()}")
except Exception as e:
    print(f"NeuroScore v2 indisponivel no notebook: {e}")
    print("Fallback para pipeline antigo (predict_brain_activation).")


# =============================================
# CONTEXTO COGNITIVO
# =============================================
CONTEXT_PROFILES = {
    "generic": {
        "label": "Genérico",
        "weights": {"v1": 1.0, "ffa": 1.0, "ppa": 1.0, "v5": 1.0, "ips": 1.0, "broca": 1.0, "pfc": 1.0, "semantic": 1.0},
    },
    "banking": {
        "label": "Banking / Fintech",
        "weights": {"v1": 1.0, "ffa": 0.5, "ppa": 1.2, "v5": 0.3, "ips": 1.0, "broca": 1.5, "pfc": 1.5, "semantic": 1.0},
    },
    "ecommerce": {
        "label": "E-commerce",
        "weights": {"v1": 1.0, "ffa": 0.8, "ppa": 1.3, "v5": 0.7, "ips": 1.2, "broca": 1.0, "pfc": 1.5, "semantic": 1.0},
    },
    "gaming": {
        "label": "Gaming / Entretenimento",
        "weights": {"v1": 0.8, "ffa": 1.2, "ppa": 0.7, "v5": 1.5, "ips": 1.0, "broca": 0.5, "pfc": 0.7, "semantic": 0.8},
    },
    "content": {
        "label": "Content / News",
        "weights": {"v1": 1.0, "ffa": 0.7, "ppa": 1.0, "v5": 0.5, "ips": 1.0, "broca": 1.5, "pfc": 0.8, "semantic": 1.5},
    },
    "saas": {
        "label": "SaaS / Produtividade",
        "weights": {"v1": 1.0, "ffa": 0.4, "ppa": 1.3, "v5": 0.5, "ips": 1.2, "broca": 1.2, "pfc": 1.3, "semantic": 1.0},
    },
    "social": {
        "label": "Social Media",
        "weights": {"v1": 0.8, "ffa": 1.5, "ppa": 0.8, "v5": 1.2, "ips": 1.0, "broca": 0.7, "pfc": 0.8, "semantic": 1.0},
    },
    "health": {
        "label": "Saúde / Health",
        "weights": {"v1": 1.0, "ffa": 0.6, "ppa": 1.0, "v5": 0.3, "ips": 1.0, "broca": 1.5, "pfc": 1.3, "semantic": 1.3},
    },
}


def normalize_context_key(contexto):
    key = str(contexto or "generic").strip().lower()
    return key if key in CONTEXT_PROFILES else "generic"


def build_context_info(context_key):
    key = normalize_context_key(context_key)
    profile = CONTEXT_PROFILES[key]
    weights = profile["weights"]
    critical = [k for k, v in sorted(weights.items(), key=lambda x: x[1], reverse=True) if v > 1.0]
    return {
        "key": key,
        "label": profile["label"],
        "weights": weights,
        "critical_regions": critical,
    }


LEGACY_SCORE_KEY_MAP = {
    "visual primario": "v1",
    "visual primário": "v1",
    "visual primário (v1)": "v1",
    "visual primario (v1)": "v1",
    "v1": "v1",
    "faces / avatares": "ffa",
    "faces / avatares (ffa)": "ffa",
    "ffa": "ffa",
    "layouts / cenas": "ppa",
    "layouts / cenas (ppa)": "ppa",
    "ppa": "ppa",
    "movimento / animacao": "v5",
    "movimento / animação": "v5",
    "movimento / animação (v5)": "v5",
    "movimento / animacao (v5)": "v5",
    "v5": "v5",
    "atencao visual": "ips",
    "atenção visual": "ips",
    "atencao visual (ips)": "ips",
    "atenção visual (ips)": "ips",
    "ips": "ips",
    "texto / labels": "broca",
    "texto / labels (broca)": "broca",
    "broca": "broca",
    "decisao / memoria": "pfc",
    "decisão / memória": "pfc",
    "decisao / memoria (pfc)": "pfc",
    "decisão / memória (pfc)": "pfc",
    "pfc": "pfc",
    "semantica / contexto": "semantic",
    "semântica / contexto": "semantic",
    "semantica / contexto (sem)": "semantic",
    "semântica / contexto (sem)": "semantic",
    "semantic": "semantic",
    "sem": "semantic",
}


def _canonical_score_key(key):
    raw = str(key or "").strip().lower()
    raw = raw.replace("á", "a").replace("à", "a").replace("â", "a").replace("ã", "a")
    raw = raw.replace("é", "e").replace("ê", "e")
    raw = raw.replace("í", "i")
    raw = raw.replace("ó", "o").replace("ô", "o").replace("õ", "o")
    raw = raw.replace("ú", "u")
    raw = raw.replace("ç", "c")
    raw = " ".join(raw.split())
    raw = raw.replace("  ", " ")
    if raw in LEGACY_SCORE_KEY_MAP:
        return LEGACY_SCORE_KEY_MAP[raw]
    for canonical in ("v1", "ffa", "ppa", "v5", "ips", "broca", "pfc", "semantic"):
        if canonical in raw:
            return canonical
    return raw


def normalize_score_keys(scores):
    normalized = {}
    for key, value in (scores or {}).items():
        canonical = _canonical_score_key(key)
        try:
            numeric = float(value)
        except Exception:
            numeric = 0.0
        if canonical in normalized:
            normalized[canonical] = max(normalized[canonical], numeric)
        else:
            normalized[canonical] = numeric
    ordered = {}
    for region in ("v1", "ffa", "ppa", "v5", "ips", "broca", "pfc", "semantic"):
        if region in normalized:
            ordered[region] = max(0.0, min(100.0, float(normalized[region])))
    for key, value in normalized.items():
        if key not in ordered:
            ordered[key] = max(0.0, min(100.0, float(value)))
    return ordered


def weighted_ux_score(scores, context_key):
    info = build_context_info(context_key)
    weights = info["weights"]
    canonical_scores = normalize_score_keys(scores)
    weighted_sum = 0.0
    total_weight = 0.0
    for region, weight in weights.items():
        score = float(canonical_scores.get(region, 0.0))
        weighted_sum += score * weight
        total_weight += weight
    return int(round(weighted_sum / total_weight)) if total_weight else 0, info


# =============================================
# RELATORIO IA COM CACHE + FALLBACK DE MODELO
# =============================================
REPORT_CACHE = {}
REPORT_CACHE_LIMIT = 120


def _cache_key(scores, tipo_entrada, context_info):
    payload = {
        "scores": {k: float(v) for k, v in sorted(scores.items())},
        "tipo": tipo_entrada,
        "context": context_info.get("key", "generic"),
        "critical": context_info.get("critical_regions", []),
    }
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _cache_put(key, value):
    if key in REPORT_CACHE:
        REPORT_CACHE[key] = value
        return
    if len(REPORT_CACHE) >= REPORT_CACHE_LIMIT:
        REPORT_CACHE.pop(next(iter(REPORT_CACHE)))
    REPORT_CACHE[key] = value


def _discover_account_models(api_key):
    try:
        r = requests.get(
            "https://api.anthropic.com/v1/models",
            headers={
                "x-api-key": api_key,
                "anthropic-version": "2023-06-01",
            },
            timeout=20,
        )
        if r.status_code >= 400:
            return []
        data = r.json()
        items = data.get("data", [])
        ids = [m.get("id") for m in items if isinstance(m, dict) and m.get("id")]
        return ids
    except Exception:
        return []


def _anthropic_models():
    env_model = (os.environ.get("ANTHROPIC_MODEL") or "").strip()
    api_key = (os.environ.get("ANTHROPIC_API_KEY") or "").strip()

    preferred = [
        "claude-sonnet-4-20250514",
        "claude-3-7-sonnet-20250219",
    ]

    account_models = _discover_account_models(api_key) if api_key else []
    ordered = []

    if env_model:
        ordered.append(env_model)

    for m in preferred:
        if m in account_models and m not in ordered:
            ordered.append(m)

    for m in account_models:
        if m not in ordered:
            ordered.append(m)

    if not ordered:
        ordered = [env_model] if env_model else ["claude-sonnet-4-20250514"]

    return ordered


def _call_anthropic(prompt):
    api_key = (os.environ.get("ANTHROPIC_API_KEY") or "").strip()
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY não definido no ambiente.")

    api_url = (os.environ.get("ANTHROPIC_API_URL") or "https://api.anthropic.com/v1/messages").strip()
    models = _anthropic_models()
    errors = []

    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }

    transient_status = {429, 500, 502, 503, 504, 529}

    for model in models:
        payload = {
            "model": model,
            "max_tokens": 900,
            "temperature": 0.3,
            "messages": [{"role": "user", "content": prompt}],
        }

        for attempt in range(3):
            try:
                resp = requests.post(api_url, headers=headers, json=payload, timeout=45)

                if resp.status_code in transient_status and attempt < 2:
                    time.sleep(1.2 * (2 ** attempt))
                    continue

                if resp.status_code >= 400:
                    body = (resp.text or "")[:320]
                    errors.append(f"{model}: HTTP {resp.status_code} - {body}")
                    break

                data = resp.json()
                parts = [p.get("text", "") for p in data.get("content", []) if p.get("type") == "text"]
                text = "\n".join([p for p in parts if p]).strip()
                if text:
                    return text, model

                errors.append(f"{model}: resposta vazia")
                break

            except Exception as e:
                if attempt < 2:
                    time.sleep(1.2 * (2 ** attempt))
                    continue
                errors.append(f"{model}: {type(e).__name__}: {e}")

    raise RuntimeError(" | ".join(errors[:3]) if errors else "Falha ao chamar Anthropic")



# =============================================
# GRAFICO
# =============================================
def gerar_grafico(scores):
    nomes = [k.split("(")[0].strip() for k in scores]
    vals = list(scores.values())
    cores = ["#D85A30" if v >= 65 else "#EF9F27" if v >= 40 else "#888780" for v in vals]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(nomes, vals, color=cores, height=0.55)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Nivel de ativacao (0-100)")
    ax.set_title("Atividade cerebral por regiao", fontweight="bold", fontsize=12)
    ax.axvline(65, color="#D85A30", linestyle="--", alpha=0.35)
    ax.axvline(40, color="#EF9F27", linestyle="--", alpha=0.35)
    for bar, v in zip(bars, vals):
        ax.text(v + 1, bar.get_y() + bar.get_height() / 2, f"{v:.0f}%", va="center", fontsize=8)
    p1 = mpatches.Patch(color="#D85A30", label="Alta (>=65)")
    p2 = mpatches.Patch(color="#EF9F27", label="Media (40-64)")
    p3 = mpatches.Patch(color="#888780", label="Baixa (<40)")
    ax.legend(handles=[p1, p2, p3], fontsize=8, loc="lower right")
    plt.tight_layout()
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmp.name, dpi=140, bbox_inches="tight")
    plt.close()
    return tmp.name


# =============================================
# RELATORIO (contextual)
# =============================================
def _deterministic_relatorio(scores, context_label, critical_txt, llm_error=None):
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    strongest = top[:3]
    weakest = sorted(scores.items(), key=lambda x: x[1])[:3]
    media = float(np.mean(list(scores.values()))) if scores else 0.0

    roi_labels = {
        "v1": "V1 (visual primário)",
        "ffa": "FFA (faces/avatares)",
        "ppa": "PPA (layouts/cenas)",
        "v5": "V5 (movimento/animação)",
        "ips": "IPS (atenção visual)",
        "broca": "Broca (texto/labels)",
        "pfc": "PFC (decisão/memória)",
        "semantic": "Semântica/contexto",
    }

    def lbl(k):
        return roi_labels.get(k, k)

    rec_map = {
        "broca": "Reescrever títulos e CTAs com linguagem direta e escaneável.",
        "pfc": "Reduzir carga de decisão: menos opções por etapa e hierarquia explícita.",
        "semantic": "Adicionar contexto claro de consequência antes de ações críticas.",
        "ips": "Reforçar foco visual com contraste, agrupamento e espaçamento.",
        "ffa": "Incluir elementos humanos quando conexão/confiança forem importantes.",
        "v5": "Adicionar microtransições e pistas de estado para fluxo mais legível.",
        "ppa": "Melhorar estrutura espacial com zonas e blocos mais consistentes.",
        "v1": "Ajustar legibilidade base (contraste, ruído visual, densidade).",
    }

    linhas = []
    linhas.append("## DIAGNOSTICO GERAL")
    linhas.append("")
    linhas.append(f"A interface apresenta ativação média de {media:.0f}/100 no contexto **{context_label}**.")
    if len(strongest) >= 2:
        linhas.append(f"Maior sustentação em **{lbl(strongest[0][0])} ({strongest[0][1]:.0f})** e **{lbl(strongest[1][0])} ({strongest[1][1]:.0f})**.")
    if len(weakest) >= 2:
        linhas.append(f"Principais fragilidades em **{lbl(weakest[0][0])} ({weakest[0][1]:.0f})** e **{lbl(weakest[1][0])} ({weakest[1][1]:.0f})**.")
    linhas.append(f"Regiões críticas para o domínio: {critical_txt}.")
    linhas.append("")

    linhas.append("## PONTOS FORTES")
    linhas.append("")
    for i, (k, v) in enumerate(strongest, 1):
        linhas.append(f"{i}. **{lbl(k)} ({v:.0f}/100)**: desempenho acima da média para essa função cognitiva.")
    linhas.append("")

    linhas.append("## PONTOS DE ATENCAO")
    linhas.append("")
    for i, (k, v) in enumerate(weakest, 1):
        linhas.append(f"{i}. **{lbl(k)} ({v:.0f}/100)**: risco de fricção UX nas tarefas dependentes dessa região.")
    linhas.append("")

    linhas.append("## RECOMENDACOES DE UX")
    linhas.append("")
    for i, (k, _v) in enumerate(weakest[:4], 1):
        linhas.append(f"{i}. {rec_map.get(k, 'Ajustar hierarquia visual e clareza da interação.')}")
    linhas.append("")

    score_geral = max(0.0, min(10.0, media / 10.0))
    linhas.append("## SCORE GERAL")
    linhas.append("")
    linhas.append(f"**{score_geral:.1f}/10**. Nota calculada pelo equilíbrio entre forças e fragilidades no contexto selecionado.")

    if llm_error:
        linhas.append("")
        linhas.append("_Observação: relatório gerado em modo robusto (fallback) por indisponibilidade/instabilidade do LLM._")

    return "\n".join(linhas)


def gerar_relatorio(scores, tipo_entrada="imagem estatica", context_info=None, features=None, evidence_by_region=None):
    context_info = context_info or build_context_info("generic")
    cache_key = _cache_key(scores, tipo_entrada, context_info)
    cached = REPORT_CACHE.get(cache_key)
    if cached:
        return {
            "text": cached,
            "source": "cache",
            "model": None,
            "error": None,
        }

    context_label = context_info.get("label", "Genérico")
    critical = context_info.get("critical_regions", [])
    critical_txt = ", ".join(critical) if critical else "nenhuma região específica"

    features = features or {}
    visual = features.get("visual", {}) or {}
    text_feat = features.get("text", {}) or {}
    observed_text = str(text_feat.get("text") or "").strip()
    observed_preview = observed_text[:500] if observed_text else "(nenhum texto OCR/transcript disponível)"

    evidence_lines = [
        f"- text_likelihood: {float(visual.get('text_likelihood', 0.0) or 0.0):.2f}",
        f"- face_likelihood: {float(visual.get('face_likelihood', 0.0) or 0.0):.2f}",
        f"- layout_likelihood: {float(visual.get('layout_likelihood', 0.0) or 0.0):.2f}",
        f"- motion: {float(visual.get('motion', 0.0) or 0.0):.2f}",
        f"- attention_salience: {float(visual.get('attention_salience', 0.0) or 0.0):.2f}",
        f"- semantic_clarity: {float(text_feat.get('semantic_clarity', 0.0) or 0.0):.2f}",
        f"- cta_strength: {float(text_feat.get('cta_strength', 0.0) or 0.0):.2f}",
    ]

    scores_txt = "\n".join([f"- {r}: {s:.1f}/100" for r, s in scores.items()])
    evidence_txt = "\n".join(evidence_lines)

    prompt = f"""Você é um especialista sênior em neurociência cognitiva aplicada a UX design.

O modelo analisou uma {tipo_entrada} no contexto {context_label} e previu as seguintes ativações cerebrais (0-100):

{scores_txt}

CONTEXTO COGNITIVO:
- Domínio: {context_label}
- Regiões mais críticas para este domínio: {critical_txt}

EVIDÊNCIAS TÉCNICAS DO PIPELINE:
{evidence_txt}

TEXTO OBSERVADO NA INTERFACE (OCR/transcript):
{observed_preview}

REGRAS OBRIGATÓRIAS:
1) NÃO inventar termos, marcas, campanhas, siglas ou textos que não estejam no bloco de texto observado.
2) Se o bloco de texto observado estiver vazio, usar linguagem genérica (ex.: rótulos, taxa, benefício), sem exemplos literais.
3) Toda afirmação deve estar ancorada nos scores e nas evidências técnicas.
4) Se faltar evidência, explicitar incerteza em vez de assumir conteúdo.

Responda em português com estas seções EXATAS:
## DIAGNOSTICO GERAL
## PONTOS FORTES
## PONTOS DE ATENCAO
## RECOMENDACOES DE UX
## SCORE GERAL

Formato técnico, objetivo e acionável."""

    try:
        text, model = _call_anthropic(prompt)

        # Guarda anti-alucinação para o caso sem OCR/transcript
        if not observed_text:
            banned_tokens = ["IOF", "NUBANK", "PIX", "INVESTBACK", "CDI", "CDB"]
            upper_text = (text or "").upper()
            if any(tok in upper_text for tok in banned_tokens):
                raise RuntimeError("LLM citou termos específicos não verificados na interface")

        required_sections = [
            "## DIAGNOSTICO GERAL",
            "## PONTOS FORTES",
            "## PONTOS DE ATENCAO",
            "## RECOMENDACOES DE UX",
            "## SCORE GERAL",
        ]
        text_upper = (text or "").upper()
        if not all(sec in text_upper for sec in required_sections):
            raise RuntimeError("LLM retornou relatório incompleto")

        _cache_put(cache_key, text)
        return {
            "text": text,
            "source": "anthropic",
            "model": model,
            "error": None,
        }
    except Exception as e:
        fallback = _deterministic_relatorio(scores, context_label, critical_txt, llm_error=str(e))
        _cache_put(cache_key, fallback)
        return {
            "text": fallback,
            "source": "fallback",
            "model": None,
            "error": str(e),
        }


# =============================================
# EXTRAIR PATH DO ARQUIVO GRADIO
# =============================================
def extrair_path(arquivo):
    if arquivo is None:
        return None
    if isinstance(arquivo, str):
        return arquivo
    if isinstance(arquivo, dict):
        return arquivo.get("path") or arquivo.get("name")
    if hasattr(arquivo, "name"):
        return arquivo.name
    return str(arquivo)


def _is_video(path):
    ext = Path(path).suffix.lower()
    return ext in {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}


def _extract_text_best_effort(path):
    if _is_video(path):
        return ""
    try:
        from PIL import Image
        import pytesseract
        txt = pytesseract.image_to_string(Image.open(path), lang=os.environ.get("OCR_LANG", "por+eng"))
        txt = (txt or "").strip()
        return txt[:1500]
    except Exception:
        return ""


def _to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def _normalize_scores(scores):
    normalized = {}
    for k, v in (scores or {}).items():
        try:
            normalized[str(k)] = max(0.0, min(100.0, float(v)))
        except Exception:
            normalized[str(k)] = 0.0
    return normalized


def _run_analysis(path, contexto="generic"):
    context_key = normalize_context_key(contexto)
    context_info = build_context_info(context_key)
    tipo = "video" if _is_video(path) else "imagem estatica"

    if HAS_NEUROSCORE_V2 and analyze_media is not None:
        t0 = time.time()
        transcript = _extract_text_best_effort(path)
        result = analyze_media(path, transcript=transcript or None)
        result["ocr_text"] = transcript
        if not isinstance(result, dict):
            result = {}
        elapsed = time.time() - t0
        result["elapsed"] = float(result.get("elapsed", elapsed))
    else:
        # Fallback para pipeline antigo
        t0 = time.time()
        scores = predict_brain_activation(path)
        elapsed = time.time() - t0
        result = {
            "scores": scores,
            "elapsed": elapsed,
            "confidence_by_region": {},
            "evidence_by_region": {},
            "features": {},
            "ocr_text": "",
        }

    scores = normalize_score_keys(_normalize_scores(result.get("scores", {})))
    result["scores"] = scores

    base_ux = result.get("ux_score")
    if base_ux is None:
        base_ux = int(round(float(np.mean(list(scores.values()))))) if scores else 0
    result["base_ux_score"] = int(round(float(base_ux)))

    weighted_ux, context_info = weighted_ux_score(scores, context_key)
    result["ux_score"] = weighted_ux
    result["context"] = context_info
    result.update(scores)

    report = gerar_relatorio(
        scores,
        tipo_entrada=tipo,
        context_info=context_info,
        features=result.get("features", {}),
        evidence_by_region=result.get("evidence_by_region", {}),
    )
    result["relatorio"] = report["text"]
    result["report_source"] = report["source"]
    result["report_model"] = report["model"]
    result["report_error"] = report["error"]

    result["confidence_by_region"] = result.get("confidence_by_region", {})
    result["evidence_by_region"] = result.get("evidence_by_region", {})
    result["features"] = result.get("features", {})

    return result


# =============================================
# FUNCAO PRINCIPAL — interface Gradio visual
# =============================================
def analisar(arquivo, contexto="generic"):
    path = extrair_path(arquivo)
    if path is None:
        return None, "Faca upload de uma imagem ou video."

    print(f"[analisar] path={path} contexto={contexto}")
    result = _run_analysis(path, contexto=contexto)
    scores = result.get("scores", {})
    elapsed = float(result.get("elapsed", 0.0))
    context_label = result.get("context", {}).get("label", "Genérico")

    grafico = gerar_grafico(scores)
    relatorio_base = result.get("relatorio") or "Sem relatório."
    relatorio = f"Contexto: {context_label}\nInferencia: {elapsed:.1f}s\n\n" + relatorio_base
    return grafico, relatorio


# =============================================
# FUNCAO JSON — endpoint para o dashboard web
# =============================================
def analisar_json(arquivo, contexto="generic"):
    path = extrair_path(arquivo)
    if path is None:
        return json.dumps({"error": "no file"}, ensure_ascii=False)

    print(f"[analisar_json] path={path} contexto={contexto}")
    result = _run_analysis(path, contexto=contexto)

    payload = {
        "scores": result.get("scores", {}),
        "relatorio": result.get("relatorio", ""),
        "elapsed": float(result.get("elapsed", 0.0)),
        "ux_score": result.get("ux_score"),
        "base_ux_score": result.get("base_ux_score"),
        "context": result.get("context", build_context_info(contexto)),
        "confidence_by_region": result.get("confidence_by_region", {}),
        "evidence_by_region": result.get("evidence_by_region", {}),
        "features": result.get("features", {}),
        "report_source": result.get("report_source"),
        "report_model": result.get("report_model"),
        "report_error": result.get("report_error"),
        "ocr_text": result.get("ocr_text", ""),
    }
    return json.dumps(_to_builtin(payload), ensure_ascii=False)


# =============================================
# PUBLICAR URL NO GIST
# =============================================
def publicar_url_gist(url):
    if not GITHUB_TOKEN or not GIST_ID:
        print("Sem GITHUB_TOKEN ou GIST_ID — publicacao no Gist ignorada.")
        return
    try:
        payload = {"files": {"gradio_url.json": {"content": json.dumps({"url": url})}}}
        resp = requests.patch(
            f"https://api.github.com/gists/{GIST_ID}",
            headers={"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"},
            json=payload,
        )
        if resp.status_code == 200:
            print(f"URL publicada no Gist: {url}")
        else:
            print(f"Erro ao atualizar Gist: {resp.status_code} {resp.text[:200]}")
    except Exception as e:
        print(f"Erro ao publicar no Gist: {e}")


# =============================================
# INTERFACE GRADIO
# =============================================
context_options = [
    ("Genérico", "generic"),
    ("Banking / Fintech", "banking"),
    ("E-commerce", "ecommerce"),
    ("Gaming / Entretenimento", "gaming"),
    ("Content / News", "content"),
    ("SaaS / Produtividade", "saas"),
    ("Social Media", "social"),
    ("Saúde / Health", "health"),
]

with gr.Blocks(title="NeuralUX — UX Brain Analyzer") as app:
    gr.Markdown("## NeuralUX — UX Brain Analyzer")
    gr.Markdown("Upload de **screenshot** ou **video de tela** para analisar a ativacao cerebral prevista.")

    with gr.Row():
        with gr.Column(scale=1):
            entrada = gr.File(label="Tela do app", file_types=[".png", ".jpg", ".jpeg", ".webp", ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"])
            contexto = gr.Dropdown(label="Contexto cognitivo", choices=context_options, value="generic")
            botao = gr.Button("Analisar", variant="primary")
            gr.Markdown("*Tempo estimado: 5-25 segundos*")
        with gr.Column(scale=2):
            grafico_out = gr.Image(label="Ativacao por regiao cerebral")
            relatorio_out = gr.Textbox(label="Relatorio de UX", lines=20)

    # fn_index=0: interface visual
    botao.click(fn=analisar, inputs=[entrada, contexto], outputs=[grafico_out, relatorio_out])

    # fn_index=1: endpoint JSON para o dashboard
    json_input = gr.File(visible=False)
    json_context = gr.Textbox(visible=False)
    json_output = gr.Textbox(visible=False)
    json_btn = gr.Button(visible=False)
    json_btn.click(fn=analisar_json, inputs=[json_input, json_context], outputs=[json_output])

# Launch e capturar a share URL corretamente
result = app.launch(share=True)

# Extrair share_url dependendo do formato de retorno
share_url = None
if isinstance(result, tuple):
    for item in result:
        if isinstance(item, str) and "gradio.live" in item:
            share_url = item
            break
elif hasattr(result, 'share_url'):
    share_url = result.share_url
if not share_url and hasattr(app, 'share_url'):
    share_url = app.share_url

if share_url:
    print(f"\nShare URL: {share_url}")
    publicar_url_gist(share_url)
else:
    print("\nNao foi possivel detectar a share URL automaticamente.")
    print("Cole a URL manualmente abaixo e rode:")
    print('  publicar_url_gist("https://XXXXX.gradio.live")')
